# Image Quality Assessment -- Training Notebook

**Pipeline stage:** 1. Image Quality Assessment (see `PROJECT_CODE.md` / `IMPLEMENTATION_PLAN.md`)
**Model:** EfficientNetB0, ImageNet-pretrained backbone, 3-class softmax head
**Classes:** `Good` / `Usable` / `Reject` (EyeQ `quality` column: 0 / 1 / 2)
**Dataset:** EyeQ, read from Google Drive (read-only)
**Training policy:** per `PROJECT_CODE.md`, all model training happens in Google Colab; this is the
official training notebook for this module.

## 1. Project Overview

This notebook is a thin orchestration layer. It does **not** reimplement any training logic --
every model, dataset, training, and evaluation step below calls directly into the existing
project modules:

| Concern | Module |
|---|---|
| Dataset loading, splitting, class weights | `image_quality_dataset.py` |
| Model architecture | `image_quality_model.py` |
| Training loop, callbacks, checkpointing, resume | `train_image_quality.py`, `training/` |
| Evaluation (metrics, confusion matrix, ROC, calibration) | `evaluate_image_quality.py`, `evaluation/` |
| Inference on individual images | `image_quality_inference.py` |
| Colab-only plumbing (GPU/mixed precision setup, checkpoint paths, export, git) | `notebooks/colab_utils.py` |

### Google Drive dataset layout (verified, read-only)

This notebook assumes your Google Drive already contains **exactly** this structure -- it does
not create, assume, or depend on any folder beyond it:

```
MyDrive/
+-- DiabeticRetinopathy/
    +-- datasets/
        +-- EyeQ/
        |   +-- raw/
        |   |   +-- train/
        |   |   |   +-- images/
        |   |   |   +-- labels.csv
        |   |   +-- test/
        |   |       +-- images/
        |   |       +-- labels.csv
        |   +-- processed/
        +-- APTOS2019/
        +-- IDRiD/
```

Only `datasets/EyeQ/raw/{train,test}` is read by this notebook. `datasets/EyeQ/processed/`,
`datasets/APTOS2019/`, and `datasets/IDRiD/` belong to other pipeline stages and are not touched
here. `datasets/EyeQ/raw` is never written to (per `PROJECT_CODE.md`'s Dataset Policy) --
checkpoints, logs, the exported model, and evaluation reports are all written to the cloned
repository on the Colab VM instead (Section 5), not back into Drive.

### Before running

1. **Runtime:** `Runtime > Change runtime type > Hardware accelerator > GPU` (T4 or better recommended).
2. Confirm the Drive layout above matches your `MyDrive/DiabeticRetinopathy/` folder.
3. Run the cells **top to bottom**. No manual edits should be required if your Drive layout
   matches; `DRIVE_PROJECT_ROOT` in Section 2 is the only line to change otherwise.

## 2. Google Drive Mount & Dataset Path Configuration

Mounts Google Drive and points the project's `EYEQ_RAW_DIR` environment variable (read by
`config.py`'s `EyeQPaths`, see `os.environ.get('EYEQ_RAW_DIR')`) at the verified
`datasets/EyeQ/raw` folder inside Drive. This is the only cell that should need editing, and
only if your Drive layout differs from the structure documented in Section 1.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

# EDIT THIS only if your Drive folder layout differs from Section 1.
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/DiabeticRetinopathy"

DRIVE_EYEQ_RAW_DIR = os.path.join(DRIVE_PROJECT_ROOT, "datasets", "EyeQ", "raw")

# Read by config.py at import time (Section 5) -- must be set before `import config`
# or anything that imports it (image_quality_dataset, train_image_quality, ...).
os.environ["EYEQ_RAW_DIR"] = DRIVE_EYEQ_RAW_DIR

print(f"EYEQ_RAW_DIR -> {DRIVE_EYEQ_RAW_DIR}")

## 3. Repository Setup

Clones this repository into the Colab VM's local disk (`/content`) and enters it -- source code
and notebooks only. The dataset stays in Drive (read-only, Section 2); checkpoints, logs, and
the exported model will be written under this cloned repository, per `config.py`'s own default
paths (see Section 5).

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
MODULE_KEY = "image_quality_assessment"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Cloning {REPO_URL} (branch={BRANCH}) into {REPO_DIR} ...")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already present at {REPO_DIR}; pulling latest {BRANCH} ...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "notebooks")):
    if path not in sys.path:
        sys.path.insert(0, path)

os.chdir(REPO_DIR)
print("Entered repository root:", os.getcwd())

## 4. Install Requirements

In [ ]:
import colab_utils

colab_utils.install_requirements(REPO_DIR)

## 5. Output Path Configuration

Resolves and verifies where this run's checkpoints, TensorBoard logs, exported model, and
evaluation report will actually be written. `IQA_MODEL_DIR` / `IQA_RESULTS_DIR` are intentionally
**not** overridden to Drive paths -- the verified Drive structure (Section 1) only provisions
`datasets/`, not `models/` or `results/`, so nothing is written back into Drive. Both fall back
to `config.py`'s own defaults, which resolve inside the just-cloned repository.

In [ ]:
from config import EYEQ_RAW_DIR, IQA_MODEL_DIR, IQA_RESULTS_DIR

print(f"EYEQ_RAW_DIR    = {EYEQ_RAW_DIR}")
print(f"IQA_MODEL_DIR   = {IQA_MODEL_DIR}")
print(f"IQA_RESULTS_DIR = {IQA_RESULTS_DIR}")

assert EYEQ_RAW_DIR == DRIVE_EYEQ_RAW_DIR, (
    f"EYEQ_RAW_DIR ({EYEQ_RAW_DIR}) does not match the Drive path configured in Section 2 "
    f"({DRIVE_EYEQ_RAW_DIR}) -- config.py may have been imported before the env var was set."
)
assert IQA_MODEL_DIR.startswith(REPO_DIR), f"Expected IQA_MODEL_DIR inside {REPO_DIR}, got {IQA_MODEL_DIR}"
assert IQA_RESULTS_DIR.startswith(REPO_DIR), f"Expected IQA_RESULTS_DIR inside {REPO_DIR}, got {IQA_RESULTS_DIR}"

print("\nDataset path resolves to Google Drive; output paths resolve inside the cloned repository.")

## 6. EyeQ Dataset Verification

Verifies the EyeQ dataset is actually present and usable *before* spending time building the
model or launching a 50-epoch run. Aborts with a clear error if `labels.csv` or the `images/`
folder is missing, or if a split ends up with zero usable images.

Reuses the project's own `image_quality_dataset._read_labels` (which already drops -- and
reports -- rows whose image file is missing) instead of reimplementing that check, so "missing
images" here is computed as `len(raw labels.csv) - len(usable rows)` from the same function the
actual training run uses.

In [ ]:
import pandas as pd

import image_quality_dataset as iqd

if not os.path.isdir(EYEQ_RAW_DIR):
    raise RuntimeError(
        f"EyeQ raw dataset directory not found at {EYEQ_RAW_DIR}. "
        "Verify DRIVE_PROJECT_ROOT in Section 2 matches your actual Drive layout "
        "(MyDrive/DiabeticRetinopathy/datasets/EyeQ/raw), then re-run from Section 2."
    )

QUALITY_CLASSES = iqd.QUALITY_CLASSES
split_reports = {}

for split in ("train", "test"):
    csv_path = os.path.join(EYEQ_RAW_DIR, split, "labels.csv")
    images_dir = os.path.join(EYEQ_RAW_DIR, split, "images")

    if not os.path.isfile(csv_path):
        raise RuntimeError(f"Dataset verification failed: missing {csv_path}.")
    if not os.path.isdir(images_dir):
        raise RuntimeError(f"Dataset verification failed: missing {images_dir}.")

    raw_df = pd.read_csv(csv_path)
    verified_df = iqd._read_labels(EYEQ_RAW_DIR, split)  # drops+reports rows with missing image files
    missing = len(raw_df) - len(verified_df)

    if len(verified_df) == 0:
        raise RuntimeError(
            f"Dataset verification failed: split '{split}' has 0 usable images "
            f"after checking for missing files under {images_dir}."
        )

    split_reports[split] = {"csv_rows": len(raw_df), "usable_rows": len(verified_df), "missing_images": missing}
    print(f"[{split}] labels.csv rows: {len(raw_df)} | usable: {len(verified_df)} | missing images: {missing}")

VAL_SPLIT = 0.15  # must match image_quality_dataset.load_eyeq_datasets's default
expected_val = round(split_reports["train"]["usable_rows"] * VAL_SPLIT)
expected_train = split_reports["train"]["usable_rows"] - expected_val
print(f"\nExpected stratified split of the train pool (val_split={VAL_SPLIT}):")
print(f"  train: ~{expected_train} images")
print(f"  val:   ~{expected_val} images")
print(f"  test (held out, never used in training): {split_reports['test']['usable_rows']} images")

### Corrupted-image spot check

A full-dataset decode of every image would cost minutes-to-tens-of-minutes for no real benefit --
per `PROJECT_CODE.md`'s verification policy, this is a lightweight spot check (a fixed random
sample per split), not a replacement for the real thing. It reuses the exact same JPEG decode
path the training pipeline uses (`image_quality_dataset._decode_image`), so a pass here means
"the images this notebook will actually feed the model decode cleanly," not just "the files
exist." Aborts if more than 20% of the sampled images in a split fail to decode.

In [ ]:
SAMPLE_SIZE = 50
CORRUPTION_ABORT_FRACTION = 0.20


def spot_check_corruption(raw_dir, split, sample_size=SAMPLE_SIZE):
    df = iqd._read_labels(raw_dir, split)
    sample = df.sample(n=min(sample_size, len(df)), random_state=42)
    failures = []
    for _, row in sample.iterrows():
        try:
            iqd._decode_image(row["path"], (224, 224)).numpy()
        except Exception as exc:  # noqa: BLE001 -- want to catch and report any decode failure
            failures.append((row["path"], str(exc)))
    return len(sample), failures


for split in ("train", "test"):
    sampled, failures = spot_check_corruption(EYEQ_RAW_DIR, split)
    print(f"[{split}] corrupted-image spot check: {len(failures)}/{sampled} sampled images failed to decode")
    for path, err in failures[:5]:
        print(f"    FAILED: {path} -- {err}")

    if sampled > 0 and len(failures) / sampled > CORRUPTION_ABORT_FRACTION:
        raise RuntimeError(
            f"Dataset verification failed: {len(failures)}/{sampled} sampled images in split "
            f"'{split}' failed to decode ({len(failures) / sampled:.0%} > "
            f"{CORRUPTION_ABORT_FRACTION:.0%} threshold). The dataset may be corrupted or "
            "incompletely uploaded to Drive."
        )

print("\nDataset verification passed.")

## 7. GPU Verification

In [ ]:
import tensorflow as tf

gpus = colab_utils.check_gpu()

## 8. TensorFlow Verification

Confirms the installed TensorFlow meets `requirements.txt`'s `tensorflow>=2.9.0` constraint --
the whole training stack (`training/`, `image_quality_model.py`) depends on Keras 3 APIs only
available from TF 2.9+.

In [ ]:
MIN_TF_VERSION = "2.9.0"


def _version_tuple(version_string):
    return tuple(int(part) for part in version_string.split(".")[:3] if part.isdigit())


print(f"TensorFlow version: {tf.__version__}")
try:
    import keras
    print(f"Keras version:      {keras.__version__}")
except ImportError:
    print("Keras version:      (standalone `keras` package not installed; tf.keras is used instead)")

if _version_tuple(tf.__version__) < _version_tuple(MIN_TF_VERSION):
    raise RuntimeError(
        f"TensorFlow {tf.__version__} is older than the minimum required {MIN_TF_VERSION} "
        "(see requirements.txt)."
    )
print(f"TensorFlow version satisfies requirements.txt's tensorflow>={MIN_TF_VERSION} constraint.")

## 9. Mixed Precision Verification

Enables `mixed_float16` when a GPU is present (falls back to `float32` on CPU-only runtimes,
where mixed precision provides no benefit). `image_quality_model.build_iqa_model` already casts
its output layer to `float32` explicitly, so this is safe to enable regardless.

In [ ]:
mixed_precision_policy = colab_utils.setup_mixed_precision()

expected_policy_name = "mixed_float16" if gpus else "float32"
assert mixed_precision_policy.name == expected_policy_name, (
    f"Unexpected mixed precision policy: {mixed_precision_policy.name} "
    f"(expected {expected_policy_name})"
)
print(f"Mixed precision policy verified: {mixed_precision_policy.name}")

## 10. Dataset Statistics

Class distribution for each split, plotted with the project's own `evaluation.plot_metric_bar`
rather than a one-off chart, so it renders identically to the bar charts produced later by the
evaluation step.

In [ ]:
from evaluation import plot_metric_bar

train_counts = pd.read_csv(os.path.join(EYEQ_RAW_DIR, "train", "labels.csv"))["quality"].value_counts()
test_counts = pd.read_csv(os.path.join(EYEQ_RAW_DIR, "test", "labels.csv"))["quality"].value_counts()

train_dist = {QUALITY_CLASSES[i]: int(train_counts.get(i, 0)) for i in range(len(QUALITY_CLASSES))}
test_dist = {QUALITY_CLASSES[i]: int(test_counts.get(i, 0)) for i in range(len(QUALITY_CLASSES))}

print("Train quality distribution:", train_dist)
print("Test quality distribution:", test_dist)

plot_metric_bar(train_dist, ylabel="Image count", title="EyeQ Train Split -- Quality Class Distribution")
plot_metric_bar(test_dist, ylabel="Image count", title="EyeQ Test Split -- Quality Class Distribution")

## 11. Sample Image Visualization

A few real images per class, decoded through the exact same `image_quality_dataset._decode_image`
path the model will see during training -- raw RGB, resized, no CLAHE/Ben Graham/normalization
(see `image_quality_dataset.py`'s module docstring: the IQA model must learn from unmodified
fundus images).

In [ ]:
import matplotlib.pyplot as plt

train_df_full = iqd._read_labels(EYEQ_RAW_DIR, "train")

fig, axes = plt.subplots(len(QUALITY_CLASSES), 4, figsize=(14, 3.2 * len(QUALITY_CLASSES)))
for row, cls_idx in enumerate(range(len(QUALITY_CLASSES))):
    cls_rows = train_df_full[train_df_full["quality"] == cls_idx].sample(n=4, random_state=42)
    for col, (_, r) in enumerate(cls_rows.iterrows()):
        image = iqd._decode_image(r["path"], (224, 224)).numpy().astype("uint8")
        ax = axes[row, col]
        ax.imshow(image)
        ax.set_title(f"{QUALITY_CLASSES[cls_idx]}\n{r['image']}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 12. Training Configuration

Matches the defaults already established in `train_image_quality.py` / `image_quality_model.py`
(EfficientNetB0, 224x224, Adam @ 1e-4, 100 frozen backbone layers, 50 epochs with early
stopping). Batch size defaults to 32, falling back to 16 automatically when the detected GPU has
less than 12 GB of memory (EfficientNetB0 at 224x224/batch 32 fits comfortably on a 16 GB T4;
smaller GPUs use the same batch size `train_image_quality.py`'s own CLI default already uses).

In [ ]:
IMAGE_SIZE = (224, 224)
EPOCHS = 50
LEARNING_RATE = 1e-4
FREEZE_LAYERS = 100
RESUME_TRAINING = False  # set True to continue from the last checkpoint of a previous run


def select_batch_size(preferred=32, fallback=16, min_total_mib=12000):
    if not gpus:
        print(f"No GPU detected -- using batch size {fallback} (CPU run).")
        return fallback
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        )
        total_mib = int(result.stdout.strip().splitlines()[0])
        print(f"Detected GPU memory: {total_mib} MiB")
        if total_mib >= min_total_mib:
            print(f"Using batch size {preferred}.")
            return preferred
        print(f"GPU memory below {min_total_mib} MiB threshold -- falling back to batch size {fallback}.")
        return fallback
    except Exception as exc:  # noqa: BLE001 -- nvidia-smi may be unavailable in some runtimes
        print(f"Could not query GPU memory ({exc}); defaulting to batch size {fallback}.")
        return fallback


BATCH_SIZE = select_batch_size()

RUN_DIR = os.path.join(IQA_MODEL_DIR, "training_run")
EXPORT_PATH = os.path.join(IQA_MODEL_DIR, "best_model.keras")

print("\nTraining configuration:")
print(f"  IMAGE_SIZE      = {IMAGE_SIZE}")
print(f"  BATCH_SIZE      = {BATCH_SIZE}")
print(f"  EPOCHS          = {EPOCHS}")
print(f"  LEARNING_RATE   = {LEARNING_RATE}")
print(f"  FREEZE_LAYERS   = {FREEZE_LAYERS}")
print(f"  RESUME_TRAINING = {RESUME_TRAINING}")
print(f"  RUN_DIR         = {RUN_DIR}")
print(f"  EXPORT_PATH     = {EXPORT_PATH}")

## 13. Model Construction

Builds the model once here purely to inspect the architecture (layer count, frozen vs.
trainable parameters) before committing to a full run. `train_image_quality.train()` in
Section 14 builds and compiles its own model instance from the same
`image_quality_model.build_iqa_model` factory -- this preview model is discarded, not reused,
so the two are never out of sync.

In [ ]:
from image_quality_model import build_iqa_model

preview_model = build_iqa_model(
    input_shape=(*IMAGE_SIZE, 3), learning_rate=LEARNING_RATE, freeze_layers=FREEZE_LAYERS,
)
preview_model.summary()

trainable_params = sum(int(tf.size(w)) for w in preview_model.trainable_weights)
total_params = preview_model.count_params()
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

del preview_model

## 14. Training

Calls `train_image_quality.train()` directly -- dataset loading, model construction, mixed
precision, checkpointing (best + last, `.keras` full-model saves under `RUN_DIR/checkpoints`),
early stopping, `ReduceLROnPlateau`, TensorBoard logging (`RUN_DIR/logs`), and resume support are
all handled inside that function and `training.Trainer`. Nothing here reimplements any of it.

In [ ]:
from train_image_quality import train

model, history, exported_path = train(
    raw_dir=EYEQ_RAW_DIR,
    run_dir=RUN_DIR,
    export_path=EXPORT_PATH,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    freeze_layers=FREEZE_LAYERS,
    resume=RESUME_TRAINING,
)

print(f"\nBest model exported to: {exported_path}")

## 15. TensorBoard

Points at the logs the `TensorBoard` callback wrote during Section 14
(`RUN_DIR/logs`, via `training.callbacks.build_callbacks`). To watch training live instead of
after the fact, this cell can be run in a separate browser tab/session before Section 14; it's
placed here so the notebook still runs unattended top-to-bottom.

In [ ]:
logs_dir = os.path.join(RUN_DIR, "logs")
%load_ext tensorboard
%tensorboard --logdir $logs_dir

## 16. Evaluation

Runs the exported model over the held-out `datasets/EyeQ/raw/test/` split -- never used for
training or validation above -- via `evaluate_image_quality.evaluate()`, which wires
`image_quality_dataset.load_eyeq_test_split` and the reusable `evaluation.Evaluator` together.
Accuracy, precision, recall, F1, AUC, quadratic weighted kappa, the confusion matrix, and the
full per-class classification report are all computed and printed by `evaluate()` itself; it
also saves `confusion_matrix.png`, `roc_curves.png`, `calibration_curve.png`, and
`evaluation_report.json` under `IQA_RESULTS_DIR`.

In [ ]:
from evaluate_image_quality import evaluate

test_report = evaluate(
    raw_dir=EYEQ_RAW_DIR,
    model_path=exported_path,
    batch_size=BATCH_SIZE,
    output_dir=IQA_RESULTS_DIR,
)

### ROC curves and calibration

In [ ]:
from IPython.display import Image, display

display(Image(filename=os.path.join(IQA_RESULTS_DIR, "roc_curves.png")))
display(Image(filename=os.path.join(IQA_RESULTS_DIR, "calibration_curve.png")))

## 17. Training Curves

Reuses `colab_utils.plot_history` (already shared across every training notebook in this repo)
rather than re-plotting `history.history` by hand.

In [ ]:
colab_utils.plot_history(history, output_path=os.path.join(RUN_DIR, "training_history.png"))

## 18. Confusion Matrix

`test_report` (from Section 16) already carries the confusion matrix and full per-class report;
the plot was already rendered to disk by `Evaluator.evaluate_and_visualize` as part of
`evaluate()` -- displayed here rather than re-plotted.

In [ ]:
display(Image(filename=os.path.join(IQA_RESULTS_DIR, "confusion_matrix.png")))

print("Confusion matrix (rows = true label, cols = predicted label):")
print(test_report.confusion_matrix)

print("\nPer-class classification report:")
pd.DataFrame(test_report.per_class_report).T

## 19. Sample Predictions

Runs `image_quality_inference.predict_quality` (the same function later pipeline stages call)
on a handful of real held-out test images, loading the exported model once and reusing it
across calls.

In [ ]:
from image_quality_inference import load_iqa_model, predict_quality

inference_model = load_iqa_model(exported_path)

test_df_full = iqd._read_labels(EYEQ_RAW_DIR, "test")
sample_rows = test_df_full.sample(n=min(8, len(test_df_full)), random_state=7)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    result = predict_quality(row["path"], model=inference_model, image_size=IMAGE_SIZE)
    image = iqd._decode_image(row["path"], IMAGE_SIZE).numpy().astype("uint8")
    true_label = QUALITY_CLASSES[int(row["quality"])]
    pred_label = result["label"]
    outcome = "MATCH" if pred_label == true_label else "MISMATCH"

    ax.imshow(image)
    ax.set_title(f"{outcome}\ntrue={true_label} pred={pred_label}\nconf={result['confidence']:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 20. Model Export

The best checkpoint was already exported to `EXPORT_PATH` -- inside the cloned repository at
`models/image_quality_assessment/best_model.keras` (Section 5) -- by
`train_image_quality.train()` in Section 14; no further copy step is needed. This matches
`PROJECT_CODE.md`'s "local repository should contain trained weights" guidance directly, with no
extra Drive folder invented for it. Committing this file to git is optional and disabled by
default; review it yourself before enabling either flag.

In [ ]:
assert os.path.exists(exported_path), "Export path missing -- did training complete?"
print(f"Best model exported to: {exported_path}")
print(f"Size: {os.path.getsize(exported_path) / 1e6:.2f} MB")

In [ ]:
DO_COMMIT_AND_PUSH = False
DO_PUSH = False

if DO_COMMIT_AND_PUSH:
    colab_utils.git_commit_and_push(
        REPO_DIR,
        message=f"Add trained {MODULE_KEY} weights",
        paths=[os.path.relpath(exported_path, REPO_DIR)],
        push=DO_PUSH,
    )
else:
    print("Skipped -- set DO_COMMIT_AND_PUSH = True to enable (see markdown above).")

## 21. Final Summary

In [ ]:
epochs_run = len(history.history["loss"])

print("=" * 72)
print("IMAGE QUALITY ASSESSMENT -- TRAINING SUMMARY")
print("=" * 72)
print(f"Model:            EfficientNetB0 (ImageNet-pretrained, first {FREEZE_LAYERS} layers frozen)")
print(f"Classes:          {QUALITY_CLASSES}")
print(f"Image size:       {IMAGE_SIZE}")
print(f"Batch size:       {BATCH_SIZE}")
print(f"Epochs run:       {epochs_run} / {EPOCHS} (early stopping may have triggered before {EPOCHS})")

print("\nFinal epoch training/validation metrics:")
for key, values in history.history.items():
    print(f"  {key}: {values[-1]:.4f}")

print("\nHeld-out test-split metrics (datasets/EyeQ/raw/test, never used in training):")
print(f"  accuracy:                 {test_report.accuracy:.4f}")
print(f"  precision (macro):        {test_report.precision:.4f}")
print(f"  recall (macro):           {test_report.recall:.4f}")
print(f"  f1 (macro):               {test_report.f1:.4f}")
print(f"  auc:                      {test_report.auc:.4f}")
print(f"  quadratic weighted kappa: {test_report.quadratic_weighted_kappa:.4f}")

print("\nArtifacts (all inside the cloned repository, not Drive):")
print(f"  Best model:                 {exported_path}")
print(f"  Evaluation report + plots:  {IQA_RESULTS_DIR}")
print(f"  TensorBoard logs:           {os.path.join(RUN_DIR, 'logs')}")
print(f"  Dataset source (read-only): {EYEQ_RAW_DIR}")

print("\nNext step in the pipeline (see PROJECT_CODE.md): Step 2 -- Image Preprocessing,")
print("gated by this model's quality predictions via image_quality_inference.py.")